## Step 1: Set Up Git Environment

GitHub was used as the Python environment for the RFID baggage tracking simulation. Google Drive was connected so the simulated data could be saved and used during the remaining analysis.

In [1]:
# Step 1: Set up the RFID simulation environment

# Load the libraries

import pandas as pd
import numpy as np

from datetime import datetime, timedelta

# Set the random state

np.random.seed(42)

print("The RFID simulation environment is ready.")

The RFID simulation environment is ready.


## Step 2: Define the Baggage Custody Flow

The normal baggage custody flow was defined for direct and transfer journeys. Direct journeys move from acceptance through claim without a transfer event. Transfer journeys include an additional transfer point before arrival.

In [40]:
# Step 2: Define the baggage custody flow

# Define the direct route

direct_flow = [
    {"sequence": 1, "custody_point": "Acceptance", "reader_location": "Check In", "reader_id": "RDR001"},
    {"sequence": 2, "custody_point": "Screening", "reader_location": "Screening Area", "reader_id": "RDR002"},
    {"sequence": 3, "custody_point": "Sorting", "reader_location": "Baggage Sort Area", "reader_id": "RDR003"},
    {"sequence": 4, "custody_point": "Loading", "reader_location": "Gate Ramp", "reader_id": "RDR004"},
    {"sequence": 5, "custody_point": "Arrival", "reader_location": "Arrival Sort Area", "reader_id": "RDR006"},
    {"sequence": 6, "custody_point": "Claim", "reader_location": "Baggage Claim", "reader_id": "RDR007"}
]

# Define the transfer route

transfer_flow = [
    {"sequence": 1, "custody_point": "Acceptance", "reader_location": "Check In", "reader_id": "RDR001"},
    {"sequence": 2, "custody_point": "Screening", "reader_location": "Screening Area", "reader_id": "RDR002"},
    {"sequence": 3, "custody_point": "Sorting", "reader_location": "Baggage Sort Area", "reader_id": "RDR003"},
    {"sequence": 4, "custody_point": "Loading", "reader_location": "Gate Ramp", "reader_id": "RDR004"},
    {"sequence": 5, "custody_point": "Transfer", "reader_location": "Transfer Area", "reader_id": "RDR005"},
    {"sequence": 6, "custody_point": "Arrival", "reader_location": "Arrival Sort Area", "reader_id": "RDR006"},
    {"sequence": 7, "custody_point": "Claim", "reader_location": "Baggage Claim", "reader_id": "RDR007"}
]

# Turn the custody flows into tables

direct_flow_df = pd.DataFrame(direct_flow)
transfer_flow_df = pd.DataFrame(transfer_flow)

# Show the custody flows

print("Direct route:")
display(direct_flow_df)

print("\nTransfer route:")
display(transfer_flow_df)

Direct route:


,sequence,custody_point,reader_location,reader_id
0,1,Acceptance,Check In,RDR001
1,2,Screening,Screening Area,RDR002
2,3,Sorting,Baggage Sort Area,RDR003
3,4,Loading,Gate Ramp,RDR004
4,5,Arrival,Arrival Sort Area,RDR006
5,6,Claim,Baggage Claim,RDR007



Transfer route:


,sequence,custody_point,reader_location,reader_id
0,1,Acceptance,Check In,RDR001
1,2,Screening,Screening Area,RDR002
2,3,Sorting,Baggage Sort Area,RDR003
3,4,Loading,Gate Ramp,RDR004
4,5,Transfer,Transfer Area,RDR005
5,6,Arrival,Arrival Sort Area,RDR006
6,7,Claim,Baggage Claim,RDR007


## Step 3: Create the Baggage Journeys

The simulated baggage journeys were created using the settings defined in Chapter 3. The simulation includes 5,000 bags. Seventy percent follow a direct route, 30 percent follow a transfer route, 70 percent remain normal, and 30 percent are assigned abnormal activity.

In [41]:
# Step 3: Create the baggage journeys

# Set the number of simulated bags

num_bags = 5000

# Set the route counts

num_direct_bags = 3500
num_transfer_bags = 1500

# Set the normal and abnormal counts

num_normal_bags = 3500
num_abnormal_bags = 1500

# Create the bag numbers

bag_numbers = np.arange(1, num_bags + 1)

# Select the transfer bags

transfer_bag_numbers = np.random.choice(
    bag_numbers,
    size=num_transfer_bags,
    replace=False
)

# Select the abnormal bags

abnormal_bag_numbers = np.random.choice(
    bag_numbers,
    size=num_abnormal_bags,
    replace=False
)

# Create an empty list for the baggage journeys

journey_records = []

# Create each baggage journey

for bag_num in bag_numbers:

    bag_id = f"BAG{bag_num:04d}"
    tag_id = f"TAG{bag_num:04d}"

    if bag_num in transfer_bag_numbers:
        route_type = "Transfer"
        transfer_status = "Yes"
    else:
        route_type = "Direct"
        transfer_status = "No"

    if bag_num in abnormal_bag_numbers:
        journey_status = "Abnormal"
    else:
        journey_status = "Normal"

    journey_records.append({
        "bag_id": bag_id,
        "tag_id": tag_id,
        "route_type": route_type,
        "transfer_status": transfer_status,
        "journey_status": journey_status
    })

# Turn the baggage journeys into a table

journey_data = pd.DataFrame(journey_records)

# Check the baggage journeys

print("Total baggage journeys:", len(journey_data))

print("\nRoute counts:")
print(journey_data["route_type"].value_counts())

print("\nJourney status:")
print(journey_data["journey_status"].value_counts())

Total baggage journeys: 5000

Route counts:
route_type
Direct      3500
Transfer    1500
Name: count, dtype: int64

Journey status:
journey_status
Normal      3500
Abnormal    1500
Name: count, dtype: int64


In [42]:
# Create the normal scan records

# Set the starting time

base_time = datetime(2026, 1, 1, 8, 0, 0)

# Create an empty list for the scan records

scan_records = []

# Create the scan records for each baggage journey

for bag_num in bag_numbers:

    bag_id = f"BAG{bag_num:04d}"
    tag_id = f"TAG{bag_num:04d}"

    # Select the correct custody flow

    if bag_num in transfer_bag_numbers:
        baggage_flow = transfer_flow
    else:
        baggage_flow = direct_flow

    # Give each bag a starting scan time

    scan_start_time = base_time + timedelta(
        minutes=np.random.randint(0, 60)
    )

    # Create one scan for each custody point

    for point in baggage_flow:

        sequence = point["sequence"]

        scan_time = scan_start_time + timedelta(
            minutes=sequence * 10
        )

        # Identify the next expected custody point

        if sequence < len(baggage_flow):
            expected_next_point = baggage_flow[sequence]["custody_point"]
        else:
            expected_next_point = "Complete"

        # Add the scan record

        scan_records.append({
            "bag_id": bag_id,
            "tag_id": tag_id,
            "reader_id": point["reader_id"],
            "reader_location": point["reader_location"],
            "custody_point": point["custody_point"],
            "timestamp": scan_time,
            "expected_sequence": sequence,
            "actual_sequence": sequence,
            "expected_next_point": expected_next_point,
            "actual_next_point": expected_next_point,
            "scan_status": "Normal"
        })

# Turn the scan records into a table

rfid_data = pd.DataFrame(scan_records)

# Check the scan records

print("Total scan records:", len(rfid_data))
print("Total baggage journeys:", rfid_data["bag_id"].nunique())

print("\nCustody points:")
print(rfid_data["custody_point"].value_counts())

Total scan records: 31500
Total baggage journeys: 5000

Custody points:
custody_point
Acceptance    5000
Screening     5000
Sorting       5000
Loading       5000
Arrival       5000
Claim         5000
Transfer      1500
Name: count, dtype: int64


## Step 4: Add the Abnormal Scan Conditions

The abnormal baggage journeys were assigned one abnormal condition using the simulation settings defined in Chapter 3. The abnormal conditions include spoofed reader activity, duplicate scans, invalid locations, out of sequence activity, delayed scans, missing scans, and denial of service disruption.

In [43]:
# Step 4: Add the abnormal scan conditions

# Create the abnormal condition list

abnormal_types = (
    ["Spoofed Reader"] * 375 +
    ["Duplicate"] * 225 +
    ["Invalid Location"] * 225 +
    ["Out of Sequence"] * 225 +
    ["Delayed"] * 225 +
    ["Missing Scan"] * 150 +
    ["Denial of Service"] * 75
)

# Randomize the abnormal conditions

np.random.shuffle(abnormal_types)

# Add the abnormal condition column

journey_data["abnormal_type"] = "None"

# Assign one condition to each abnormal journey

journey_data.loc[
    journey_data["journey_status"] == "Abnormal",
    "abnormal_type"
] = abnormal_types

# Check the results

print("Abnormal condition counts:")

print(
    journey_data[
        journey_data["journey_status"] == "Abnormal"
    ]["abnormal_type"].value_counts()
)

print(
    "\nNormal journeys:",
    (journey_data["abnormal_type"] == "None").sum()
)

Abnormal condition counts:
abnormal_type
Spoofed Reader       375
Delayed              225
Duplicate            225
Out of Sequence      225
Invalid Location     225
Missing Scan         150
Denial of Service     75
Name: count, dtype: int64

Normal journeys: 3500


In [44]:
# Apply the abnormal conditions to the scan records

# Add the journey information

journey_info = journey_data[
    [
        "bag_id",
        "route_type",
        "transfer_status",
        "journey_status",
        "abnormal_type"
    ]
].copy()

journey_info = journey_info.rename(
    columns={"abnormal_type": "journey_abnormal_type"}
)

rfid_data = rfid_data.merge(
    journey_info,
    on="bag_id",
    how="left"
)

# Keep the expected scan values

rfid_data["expected_reader_id"] = rfid_data["reader_id"]
rfid_data["expected_reader_location"] = rfid_data["reader_location"]
rfid_data["expected_timestamp"] = pd.to_datetime(rfid_data["timestamp"])

rfid_data["timestamp"] = pd.to_datetime(rfid_data["timestamp"])

# Create lists for duplicate and missing records

duplicate_records = []
missing_rows = []

# Apply one abnormal condition to each abnormal journey

abnormal_journeys = journey_data[
    journey_data["journey_status"] == "Abnormal"
]

for bag_id in abnormal_journeys["bag_id"]:

    abnormal_type = abnormal_journeys.loc[
        abnormal_journeys["bag_id"] == bag_id,
        "abnormal_type"
    ].iloc[0]

    bag_rows = rfid_data[
        rfid_data["bag_id"] == bag_id
    ].index

    row_index = np.random.choice(bag_rows)

    if abnormal_type == "Spoofed Reader":

        rfid_data.loc[row_index, "reader_id"] = "RDR999"
        rfid_data.loc[row_index, "scan_status"] = "Spoofed Reader"

    elif abnormal_type == "Invalid Location":

        rfid_data.loc[row_index, "reader_location"] = "Unknown Zone"
        rfid_data.loc[row_index, "scan_status"] = "Invalid Location"

    elif abnormal_type == "Delayed":

        rfid_data.loc[row_index, "timestamp"] = (
            rfid_data.loc[row_index, "timestamp"]
            + pd.Timedelta(minutes=20)
        )

        rfid_data.loc[row_index, "scan_status"] = "Delayed"

    elif abnormal_type == "Out of Sequence":

        expected_sequence = rfid_data.loc[
            row_index,
            "expected_sequence"
        ]

        if expected_sequence == 1:
            rfid_data.loc[row_index, "actual_sequence"] = 2
        else:
            rfid_data.loc[row_index, "actual_sequence"] = expected_sequence - 1

        rfid_data.loc[row_index, "scan_status"] = "Out of Sequence"

    elif abnormal_type == "Duplicate":

        duplicate_record = rfid_data.loc[row_index].copy()

        duplicate_record["timestamp"] = (
            duplicate_record["timestamp"]
            + pd.Timedelta(seconds=30)
        )

        duplicate_record["scan_status"] = "Duplicate"

        duplicate_records.append(duplicate_record)

    elif abnormal_type == "Missing Scan":

        missing_rows.append(row_index)

    elif abnormal_type == "Denial of Service":

        rfid_data.loc[row_index, "scan_status"] = "Denial of Service"

# Add the duplicate records

rfid_data = pd.concat(
    [rfid_data, pd.DataFrame(duplicate_records)],
    ignore_index=True
)

# Remove the missing records

rfid_data = rfid_data.drop(
    index=missing_rows
).reset_index(drop=True)

# Check the changes

print("Abnormal journeys processed:", len(abnormal_journeys))
print("Duplicate records added:", len(duplicate_records))
print("Missing records removed:", len(missing_rows))

Abnormal journeys processed: 1500
Duplicate records added: 225
Missing records removed: 150


In [45]:
# Create the denial of service disruption windows

# Find the bags assigned to denial of service

dos_bag_ids = journey_data[
    journey_data["abnormal_type"] == "Denial of Service"
]["bag_id"].tolist()

# Find the denial of service scan for each bag

dos_rows = []

for bag_id in dos_bag_ids:

    bag_dos_row = rfid_data[
        (rfid_data["bag_id"] == bag_id) &
        (rfid_data["scan_status"] == "Denial of Service")
    ].index

    if len(bag_dos_row) > 0:
        dos_rows.append(bag_dos_row[0])

# Create the disruption fields

rfid_data["disruption_id"] = "None"
rfid_data["disruption_window"] = 0

# Assign the 75 scans across 10 disruption windows

for record_num, row_index in enumerate(dos_rows):

    window_num = (record_num % 10) + 1
    disruption_id = f"DOS{window_num:02d}"

    rfid_data.loc[row_index, "disruption_id"] = disruption_id
    rfid_data.loc[row_index, "disruption_window"] = 1

# Set the duration for each disruption window

dos_window_records = []

for window_num in range(1, 11):

    disruption_id = f"DOS{window_num:02d}"
    window_minutes = np.random.randint(15, 31)

    dos_window_records.append({
        "disruption_id": disruption_id,
        "window_minutes": window_minutes
    })

dos_windows = pd.DataFrame(dos_window_records)

# Check the denial of service conditions

print("DoS disruption windows:", len(dos_windows))
print("DoS journeys:", len(dos_bag_ids))
print("DoS scans:", (rfid_data["disruption_window"] == 1).sum())

print("\nWindow durations:")
print(dos_windows)

DoS disruption windows: 10
DoS journeys: 75
DoS scans: 75

Window durations:
  disruption_id  window_minutes
0         DOS01              20
1         DOS02              16
2         DOS03              27
3         DOS04              29
4         DOS05              20
5         DOS06              17
6         DOS07              21
7         DOS08              17
8         DOS09              23
9         DOS10              17


## Step 5: Validate and Save the RFID Data

The completed RFID baggage tracking simulation was reviewed to confirm the expected number of baggage journeys, scan records, abnormal conditions, and denial of service records. Simulated flight and route identifiers were added before the final dataset was saved for the remaining analysis.

In [46]:
# Step 5: Validate and save the RFID data

# Add the simulated route information

rfid_data["flight_id"] = "FLT001"
rfid_data["route_id"] = "RT001"

rfid_data.loc[
    rfid_data["route_type"] == "Transfer",
    "flight_id"
] = "FLT002"

rfid_data.loc[
    rfid_data["route_type"] == "Transfer",
    "route_id"
] = "RT002"

rfid_data["origin"] = "SIM_ORIGIN"
rfid_data["destination"] = "SIM_DESTINATION"

# Add a unique scan ID

scan_ids = []

for scan_num in range(1, len(rfid_data) + 1):
    scan_ids.append(f"SCAN{scan_num:05d}")

rfid_data["scan_id"] = scan_ids

# Add the disruption duration

rfid_data["disruption_minutes"] = 0

for window_num in range(1, 11):

    disruption_id = f"DOS{window_num:02d}"

    window_minutes = dos_windows.loc[
        dos_windows["disruption_id"] == disruption_id,
        "window_minutes"
    ].iloc[0]

    rfid_data.loc[
        rfid_data["disruption_id"] == disruption_id,
        "disruption_minutes"
    ] = window_minutes

# Check the final dataset

print("Total RFID records:", len(rfid_data))
print("Total baggage journeys:", rfid_data["bag_id"].nunique())
print("Unique scan IDs:", rfid_data["scan_id"].nunique())

print("\nRoute counts:")
print(
    rfid_data.drop_duplicates("bag_id")[
        "route_type"
    ].value_counts()
)

print("\nAbnormal journey counts:")
print(
    journey_data[
        journey_data["journey_status"] == "Abnormal"
    ]["abnormal_type"].value_counts()
)

print(
    "\nDoS scans:",
    (rfid_data["disruption_window"] == 1).sum()
)

# Save the RFID dataset

main_folder = "/content/drive/MyDrive/Praxis_Coding"

rfid_folder = (
    main_folder +
    "/01_Simulated_RFID_Data"
)

rfid_file = (
    rfid_folder +
    "/01_Simulated_RFID_Data.csv"
)

rfid_data.to_csv(
    rfid_file,
    index=False
)

print("\nRFID data saved:")
print(rfid_file)

Total RFID records: 31575
Total baggage journeys: 5000
Unique scan IDs: 31575

Route counts:
route_type
Direct      3500
Transfer    1500
Name: count, dtype: int64

Abnormal journey counts:
abnormal_type
Spoofed Reader       375
Delayed              225
Duplicate            225
Out of Sequence      225
Invalid Location     225
Missing Scan         150
Denial of Service     75
Name: count, dtype: int64

DoS scans: 75

RFID data saved:
/content/drive/MyDrive/Praxis_Coding/01_Simulated_RFID_Data/01_Simulated_RFID_Data.csv
